# OpenVLA Fine-tuning on AWS Trainium with SageMaker

This notebook orchestrates an end-to-end pipeline for fine-tuning OpenVLA on LIBERO tasks using AWS Trainium.

**Pipeline Steps:**
1. Prepare LIBERO data and upload to S3
2. Run ahead-of-time compilation job (short ~100 steps) to cache compiled graphs
3. Run full training job using cached compilation

**Prerequisites:**
- SageMaker execution role with S3 access
- S3 bucket for data and model artifacts
- Quota for ml.trn1.32xlarge instances

## 1. Configuration

In [ ]:
import sagemaker
from sagemaker.pytorch import PyTorch
import boto3

# SageMaker session and role
sess = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sess.boto_region_name

# S3 configuration
S3_BUCKET = sess.default_bucket()  # Or specify your bucket
S3_PREFIX = "openvla-trainium"
S3_DATA_URI = f"s3://{S3_BUCKET}/{S3_PREFIX}/data"
S3_NEURON_CACHE = f"s3://{S3_BUCKET}/{S3_PREFIX}/neuron-cache"
S3_OUTPUT = f"s3://{S3_BUCKET}/{S3_PREFIX}/output"

# Neuron DLC image (PyTorch 2.1 Neuron)
# See: https://github.com/aws/deep-learning-containers/blob/master/available_images.md
NEURON_IMAGE = f"763104351884.dkr.ecr.{region}.amazonaws.com/pytorch-training-neuronx:2.1.2-neuronx-py310-sdk2.20.2-ubuntu20.04"

print(f"S3 Bucket: {S3_BUCKET}")
print(f"Data URI: {S3_DATA_URI}")
print(f"Neuron Cache: {S3_NEURON_CACHE}")
print(f"Output: {S3_OUTPUT}")

## 2. Prepare Data

Download LIBERO dataset from HuggingFace and upload to S3.

In [ ]:
# Option A: Run data preparation (downloads ~2GB)
# Uncomment to download and upload data

# !pip install huggingface_hub
# !git lfs install
# !python prepare_data.py --s3-bucket {S3_BUCKET} --s3-prefix {S3_PREFIX}/data

In [ ]:
# Option B: Use existing data in S3
# Set this to your data location if already uploaded

# Verify data exists
!aws s3 ls {S3_DATA_URI}/ --summarize | tail -3

## 3. Training Hyperparameters

In [ ]:
# Common hyperparameters
HYPERPARAMETERS = {
    "task_suites": "libero_spatial",
    "batch_size": 1,
    "learning_rate": 2e-5,
    "lora_rank": 32,
    "tensor_parallel_size": 8,
    "save_interval": 5000,
    "vla_path": "openvla/openvla-7b",
}

# Environment variables for Neuron
ENV_VARS = {
    "NEURON_COMPILE_CACHE_URL": S3_NEURON_CACHE,
    "NEURON_RT_NUM_CORES": "32",
    "TOKENIZERS_PARALLELISM": "false",
    "FI_EFA_USE_DEVICE_RDMA": "1",
    "FI_PROVIDER": "efa",
}

## 4. Ahead-of-Time Compilation Job

Run a short training job (~100 steps) with `RUN_NEURON_PARALLEL_COMPILE=1` to extract and compile all XLA graphs. The compiled NEFFs are cached to S3.

In [ ]:
# Compilation job hyperparameters (short run)
compile_hyperparams = HYPERPARAMETERS.copy()
compile_hyperparams["max_steps"] = 100  # Short run for graph extraction

# Enable parallel compilation
compile_env = ENV_VARS.copy()
compile_env["RUN_NEURON_PARALLEL_COMPILE"] = "1"

compile_estimator = PyTorch(
    entry_point="sagemaker_train.py",
    source_dir=".",  # Directory containing training scripts
    role=role,
    instance_count=1,
    instance_type="ml.trn1.32xlarge",
    image_uri=NEURON_IMAGE,
    hyperparameters=compile_hyperparams,
    environment=compile_env,
    output_path=f"{S3_OUTPUT}/compilation",
    base_job_name="openvla-compile",
    max_run=3600,  # 1 hour max for compilation
    keep_alive_period_in_seconds=0,
)

print("Starting compilation job...")
print(f"Neuron cache will be saved to: {S3_NEURON_CACHE}")

In [ ]:
# Start compilation job
compile_estimator.fit(
    inputs={"training": S3_DATA_URI},
    wait=True,
    logs="All"
)

print(f"\nCompilation complete! Cache saved to: {S3_NEURON_CACHE}")

In [ ]:
# Verify compilation cache was created
!aws s3 ls {S3_NEURON_CACHE}/ --summarize | tail -5

## 5. Full Training Job

Run the full training job using the cached compilation. This job will skip compilation and start training immediately.

In [ ]:
# Full training hyperparameters
train_hyperparams = HYPERPARAMETERS.copy()
train_hyperparams["max_steps"] = 50000  # Full training run

# No RUN_NEURON_PARALLEL_COMPILE - uses cached compilation
train_env = ENV_VARS.copy()

train_estimator = PyTorch(
    entry_point="sagemaker_train.py",
    source_dir=".",
    role=role,
    instance_count=1,
    instance_type="ml.trn1.32xlarge",
    image_uri=NEURON_IMAGE,
    hyperparameters=train_hyperparams,
    environment=train_env,
    output_path=f"{S3_OUTPUT}/training",
    base_job_name="openvla-train",
    max_run=86400,  # 24 hours max
    checkpoint_s3_uri=f"{S3_OUTPUT}/checkpoints",
    checkpoint_local_path="/opt/ml/checkpoints",
)

print("Starting training job...")
print(f"Using cached compilation from: {S3_NEURON_CACHE}")

In [ ]:
# Start training job
train_estimator.fit(
    inputs={"training": S3_DATA_URI},
    wait=False,  # Don't wait - monitor separately
    logs="None"
)

print(f"\nTraining job started: {train_estimator.latest_training_job.name}")
print(f"Monitor in SageMaker console or run: train_estimator.logs()")

## 6. Monitor Training

In [ ]:
# Check training job status
train_estimator.latest_training_job.describe()["TrainingJobStatus"]

In [ ]:
# Stream logs (blocking)
# train_estimator.logs()

In [ ]:
# Wait for completion
# train_estimator.latest_training_job.wait(logs="All")

## 7. Retrieve Results

In [ ]:
# List output artifacts
!aws s3 ls {S3_OUTPUT}/training/ --recursive | head -20

In [ ]:
# Download model artifacts
# !aws s3 cp {train_estimator.model_data} ./model.tar.gz
# !tar -xzf model.tar.gz

## Cost Estimates

| Job Type | Instance | Duration | Est. Cost |
|----------|----------|----------|----------|
| Compilation | ml.trn1.32xlarge | ~30 min | ~$10 |
| Training (50k steps) | ml.trn1.32xlarge | ~8 hours | ~$160 |

*Prices based on us-west-2 on-demand pricing. Actual costs may vary.*